In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, current_timestamp
from delta.tables import DeltaTable

CATALOG = "incremental_catalog"
SCHEMA = "pipeline_schema"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_orders"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_orders"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_orders_summary"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.quarantine_orders"
AUDIT_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_audit"
TRACKING_TABLE = f"{CATALOG}.{SCHEMA}.processed_batches"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/order_data_volume/"

print("Automation configuration loaded successfully.")

Automation configuration loaded successfully.


In [0]:
files = dbutils.fs.ls(VOLUME_PATH)

csv_files = [
    file.name
    for file in files
    if file.name.lower().endswith(".csv")
]

print("CSV files found:")
for file_name in csv_files:
    print(file_name)

print("Total CSV files:", len(csv_files))

CSV files found:
batch_1.csv
batch_2.csv
batch_3.csv
batch_4.csv
batch_5.csv
Total CSV files: 5


In [0]:
# Get successfully processed source files
processed_files = {
    row["source_file"]
    for row in (
        spark.table(TRACKING_TABLE)
        .filter(col("status") == "SUCCESS")
        .select("source_file")
        .distinct()
        .collect()
    )
}

# Identify files that have not been processed
unprocessed_files = [
    file_name
    for file_name in csv_files
    if file_name not in processed_files
]

print("Already processed:", sorted(processed_files))
print("New files detected:", unprocessed_files)
print("New file count:", len(unprocessed_files))

Already processed: ['batch_1.csv', 'batch_2.csv', 'batch_3.csv', 'batch_4.csv']
New files detected: ['batch_5.csv']
New file count: 1


In [0]:
def generate_batch_id(file_name):

    file_number = (
        file_name
        .replace("batch_", "")
        .replace(".csv", "")
    )

    return f"BATCH_{int(file_number):03d}"

In [0]:
for file_name in csv_files:
    print(file_name, "→", generate_batch_id(file_name))

batch_1.csv → BATCH_001
batch_2.csv → BATCH_002
batch_3.csv → BATCH_003
batch_4.csv → BATCH_004
batch_5.csv → BATCH_005


In [0]:
PIPELINE_NOTEBOOK_PATH = "/Workspace/Users/muflihamustafa@gmail.com/Incremental_Pipeline_Final"

print("Pipeline notebook path configured:")
print(PIPELINE_NOTEBOOK_PATH)

Pipeline notebook path configured:
/Workspace/Users/muflihamustafa@gmail.com/Incremental_Pipeline_Final


In [0]:
def execute_pipeline(batch_id, file_name):

    print(f"Executing pipeline for {batch_id}")

    result = dbutils.notebook.run(
        PIPELINE_NOTEBOOK_PATH,
        timeout_seconds=0,
        arguments={
            "batch_id": batch_id,
            "file_name": file_name
        }
    )

    print(f"Pipeline result for {batch_id}: {result}")

    return result

In [0]:
for file_name in unprocessed_files:
    batch_id = generate_batch_id(file_name)

    print(f"New file detected: {file_name}")
    print(f"Generated batch ID: {batch_id}")

    result = execute_pipeline(batch_id, file_name)

    print(f"Final result for {batch_id}: {result}")

New file detected: batch_5.csv
Generated batch ID: BATCH_005
Executing pipeline for BATCH_005


---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-5395062764755543>, line 7
      4 print(f"New file detected: {file_name}")
      5 print(f"Generated batch ID: {batch_id}")
----> 7 result = execute_pipeline(batch_id, file_name)
      9 print(f"Final result for {batch_id}: {result}")

File <command-8438378311069507>, line 5, in execute_pipeline(batch_id, file_name)
      1 def execute_pipeline(batch_id, file_name):
      3     print(f"Executing pipeline for {batch_id}")
----> 5     result = dbutils.notebook.run(
      6         PIPELINE_NOTEBOOK_PATH,
      7         timeout_seconds=0,
      8         arguments={
      9             "batch_id": batch_id,
     10             "file_name": file_name
     11         }
     12     )
     14     print(f"Pipeline result for {batch_id}: {result}")
     16     return result

File /databricks/python_shell/lib/dbruntime/dbutils.py:3

In [0]:
%sql
SELECT *
FROM incremental_catalog.pipeline_schema.pipeline_audit
WHERE batch_id = 'BATCH_005';